## 1. Answer the questions

- Derive an analytical solution to the regression problem. Use a vector form of the equation.

Запишем модель регрессии:
$$
y = Xw + \varepsilon
$$
Запишем функцию ошибки (RSS):
$$
L(w) = ||y - Xw||^2
$$
Норму в квадрате можно выразить как произведение транспонированного вектора ошибки на вектор ошибки:
$$
L(w) = (y - Xw)^T (y - Xw)
$$
Задача состоит в том, чтобы минимизировать функцию потерь. Для этого дифференцируем $L(w)$ по $w$ и приравниваем к нулю:
$$
\frac{\partial L}{\partial w} = -2X^T(y - Xw)
$$
$$
X^T X w = X^T y
$$
$$
w = (X^T X)^{-1} X^T y
$$

- What changes in the solution when L1 and L2 regularizations are added to the loss function.

### Найдем аналитическое решение с L2-регуляризацией

Функция ошибки (функция потерь + регуляризация):
$$
L(w) = \|y - Xw\|^2 + \lambda \|w\|_2^2
$$
То есть:
$$
L(w) = \|y - Xw\|^2 + \lambda w^T w
$$
Далее также дифференцируем $L(w)$ по $w$. Таким образом, решение меняется так:
$$
w = (X^T X + \lambda I)^{-1} X^T y
$$

### Решение для L1-регуляризации

Вывести аналитическое решение в таком случае нельзя. Очень часто оптимум может оказаться в точке $w_j = 0$. А в этой точки у $|w|$ угол, в которой невозможно определить производную.

- Explain why L1 regularization is often used to select features. Why are there many weights equal to 0 after the model is fit?

Объяснение по картинке:

Ромб на картинке - это границы регуляризации. По сути чем больше будет $ \lambda$, тем меньше будет этот ромб. Точка, в которой овал в первый раз касается ромба - это оптимум с минимальной функцией потерь при данной регуляризации. Очень часто овал впервые касается ромба в его вершинах, где некоторые $w_j = 0$, отсюда и выходят нулевые веса.
![plot](./images/l1_l2.png)

- Explain how you can use the same models (Linear regression, Ridge, etc.) but make it possible to fit nonlinear dependencies.

Нелинейности по $x$ можно добиться с помощью полиномиальных признаков $[x, x^2, x^3, x^4]$, а также с помощью взаимодействия признаков, к примеру, $x_1 x_2$.

## 2. Introduction

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures

In [2]:
train_data = pd.read_json("data/train.json")
test_data = pd.read_json("data/test.json")

## 3. Data Analysis (Preprocessing)

In [3]:
features = []
for value in train_data["features"]:
    for v in value:
        features.append(v)

In [4]:
counter = Counter(features)
top_20 = counter.most_common(20)
for top in top_20:
    print(f"{top}")

('Elevator', 25915)
('Cats Allowed', 23540)
('Hardwood Floors', 23527)
('Dogs Allowed', 22035)
('Doorman', 20898)
('Dishwasher', 20426)
('No Fee', 18062)
('Laundry in Building', 16344)
('Fitness Center', 13252)
('Pre-War', 9148)
('Laundry in Unit', 8738)
('Roof Deck', 6542)
('Outdoor Space', 5268)
('Dining Room', 5136)
('High Speed Internet', 4299)
('Balcony', 2992)
('Swimming Pool', 2730)
('Laundry In Building', 2593)
('New Construction', 2559)
('Terrace', 2283)


In [5]:
new_features = [top[0] for top in top_20]

for feature in new_features:
    train_data[feature] = train_data["features"].apply(
        lambda x: 1 if feature in x else 0
    )

for feature in new_features:
    test_data[feature] = test_data["features"].apply(
        lambda x: 1 if feature in x else 0
    )

In [6]:
train_data.info()

<class 'pandas.DataFrame'>
Index: 49352 entries, 4 to 124009
Data columns (total 35 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   bathrooms            49352 non-null  float64
 1   bedrooms             49352 non-null  int64  
 2   building_id          49352 non-null  str    
 3   created              49352 non-null  str    
 4   description          49352 non-null  str    
 5   display_address      49352 non-null  str    
 6   features             49352 non-null  object 
 7   latitude             49352 non-null  float64
 8   listing_id           49352 non-null  int64  
 9   longitude            49352 non-null  float64
 10  manager_id           49352 non-null  str    
 11  photos               49352 non-null  object 
 12  price                49352 non-null  int64  
 13  street_address       49352 non-null  str    
 14  interest_level       49352 non-null  str    
 15  Elevator             49352 non-null  int64  
 16  C

- избавляемся от выбросов в данных

In [7]:
train_data = train_data[new_features + ["bathrooms", "bedrooms", "price"]]

p1 = np.percentile(train_data["price"], 1)
p99 = np.percentile(train_data["price"], 99)

train_data = train_data.loc[(train_data["price"] >= p1) & (train_data["price"] <= p99), :]

test_data = test_data[new_features + ["bathrooms", "bedrooms", "price"]]
test_data = test_data.loc[(test_data["price"] >= p1) & (test_data["price"] <= p99), :]

- формируем X_train, y_train, X_test, y_test

In [8]:
X_train = train_data.drop(columns=["price"])
y_train = train_data["price"]

X_test = test_data.drop(columns=["price"])
y_test = test_data["price"]

X_train.info()

<class 'pandas.DataFrame'>
Index: 48379 entries, 4 to 124009
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Elevator             48379 non-null  int64  
 1   Cats Allowed         48379 non-null  int64  
 2   Hardwood Floors      48379 non-null  int64  
 3   Dogs Allowed         48379 non-null  int64  
 4   Doorman              48379 non-null  int64  
 5   Dishwasher           48379 non-null  int64  
 6   No Fee               48379 non-null  int64  
 7   Laundry in Building  48379 non-null  int64  
 8   Fitness Center       48379 non-null  int64  
 9   Pre-War              48379 non-null  int64  
 10  Laundry in Unit      48379 non-null  int64  
 11  Roof Deck            48379 non-null  int64  
 12  Outdoor Space        48379 non-null  int64  
 13  Dining Room          48379 non-null  int64  
 14  High Speed Internet  48379 non-null  int64  
 15  Balcony              48379 non-null  int64  
 16  S

## 4. Models implementation — Linear regression

Deterministic model — это модель, которая при одинаковых входных данных всегда даёт одинаковый результат.
Чтобы сделать SGD детерминированным, нужно зафиксировать генератор случайных чисел (seed).

In [9]:
class MyLinearRegression:
    def __init__(self, method="sgd", lr=0.001, n_iter=1000, seed=42):
        self.method = method
        self.lr = lr
        self.n_iter = n_iter
        self.seed = seed
        self.w = None

    def add_bias(self, X):
        return np.c_[np.ones(X.shape[0]), X]

    def fit(self, X, y):
        X = self.add_bias(X)
        n_samples, n_features = X.shape

        if self.method == "analytical":
            self.w = np.linalg.inv(X.T @ X) @ X.T @ y

        elif self.method == "gd":
            self.w = np.zeros(n_features)

            for _ in range(self.n_iter):
                y_pred = X @ self.w
                gradient = (2 / n_samples) * X.T @ (y_pred - y)
                self.w -= self.lr * gradient

        elif self.method == "sgd":
            rng = np.random.default_rng(self.seed)
            self.w = np.zeros(n_features)

            if isinstance(X, pd.DataFrame):
                X = X.to_numpy()

            if isinstance(y, pd.Series):
                y = y.to_numpy()

            for _ in range(self.n_iter):
                for i in rng.permutation(n_samples):
                    xi = X[i]
                    yi = y[i]

                    y_pred = xi @ self.w
                    gradient = 2 * xi * (y_pred - yi)

                    self.w -= self.lr * gradient

        return self

    def predict(self, X):
        X = self.add_bias(X)
        return X @ self.w

Написала функции для подсчета метрик:
$$
MAE = \frac{1}{N} \sum_{i=1}^{N} |y_i - \hat{y}_i|
$$
$$
RMSE = \sqrt{ \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2 }
$$
$$
R^2 = 1 - \frac{\sum_{i=1}^{N} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{N} (y_i - \bar{y})^2}
$$

In [10]:
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot


def initialize_metric_tables():
    columns = ["model", "train", "test"]
    return {
        "MAE": pd.DataFrame(columns=columns),
        "RMSE": pd.DataFrame(columns=columns),
        "R2": pd.DataFrame(columns=columns),
    }


def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    return {
        "MAE": (mae(y_train, y_train_pred), mae(y_test, y_test_pred)),
        "RMSE": (rmse(y_train, y_train_pred), rmse(y_test, y_test_pred)),
        "R2": (r2_score(y_train, y_train_pred), r2_score(y_test, y_test_pred)),
    }


def add_metrics(results, model_name, metrics):
    for metric_name, (train_value, test_value) in metrics.items():
        results[metric_name].loc[len(results[metric_name])] = [
            model_name,
            train_value,
            test_value,
        ]
    return results


def fit_evaluate_add(results, model_name, model, X_train, y_train, X_test, y_test):
    metrics = evaluate_model(model, X_train, y_train, X_test, y_test)
    add_metrics(results, model_name, metrics)
    return metrics


def sync_metric_tables(results):
    return results["MAE"], results["RMSE"], results["R2"]


def select_metric_rows(results, model_names):
    selected = {}
    for metric_name, table in results.items():
        selected[metric_name] = table.loc[
            table["model"].isin(model_names), :
        ].reset_index(drop=True)
    return selected


def scale_selected_columns(X_train, X_test, scaler, columns):
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    X_train_scaled[columns] = scaler.fit_transform(X_train[columns])
    X_test_scaled[columns] = scaler.transform(X_test[columns])
    return X_train_scaled, X_test_scaled


def make_polynomial_features(X_train, X_test, columns, degree=10):
    polynomial_features = PolynomialFeatures(degree=degree, include_bias=False)

    poly_train = polynomial_features.fit_transform(X_train[columns])
    poly_train = pd.DataFrame(
        poly_train,
        columns=polynomial_features.get_feature_names_out(columns),
        index=X_train.index,
    )

    poly_test = polynomial_features.transform(X_test[columns])
    poly_test = pd.DataFrame(
        poly_test,
        columns=polynomial_features.get_feature_names_out(columns),
        index=X_test.index,
    )

    X_train_poly = pd.concat([X_train.drop(columns=columns), poly_train], axis=1)
    X_test_poly = pd.concat([X_test.drop(columns=columns), poly_test], axis=1)
    return X_train_poly, X_test_poly, polynomial_features


def scale_non_binary_features(X_train, X_test, scaler):
    binary_cols = [
        col for col in X_train.columns
        if set(pd.Series(X_train[col]).dropna().unique()) <= {0, 1}
    ]
    num_cols = [col for col in X_train.columns if col not in binary_cols]

    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])
    return X_train_scaled, X_test_scaled, binary_cols, num_cols


def add_baseline_result(results, model_name, train_value, test_value, y_train, y_test):
    y_train_pred = pd.Series(train_value, index=y_train.index)
    y_test_pred = pd.Series(test_value, index=y_test.index)
    metrics = {
        "MAE": (mae(y_train, y_train_pred), mae(y_test, y_test_pred)),
        "RMSE": (rmse(y_train, y_train_pred), rmse(y_test, y_test_pred)),
        "R2": (r2_score(y_train, y_train_pred), r2_score(y_test, y_test_pred)),
    }
    add_metrics(results, model_name, metrics)
    return metrics

In [11]:
results = initialize_metric_tables()
MAE, RMSE, R2 = sync_metric_tables(results)
results

{'MAE': Empty DataFrame
 Columns: [model, train, test]
 Index: [],
 'RMSE': Empty DataFrame
 Columns: [model, train, test]
 Index: [],
 'R2': Empty DataFrame
 Columns: [model, train, test]
 Index: []}

In [12]:
linear_methods = ["sgd", "gd", "analytical"]

for method in linear_methods:
    model = MyLinearRegression(method=method, lr=0.001, n_iter=100)
    fit_evaluate_add(
        results,
        f"{model.__class__.__name__}_({method})",
        model,
        X_train,
        y_train,
        X_test,
        y_test,
    )

MAE, RMSE, R2 = sync_metric_tables(results)

In [13]:
fit_evaluate_add(
    results,
    "LinearRegression_(sklearn)",
    LinearRegression(),
    X_train,
    y_train,
    X_test,
    y_test,
)

MAE, RMSE, R2 = sync_metric_tables(results)
RMSE

,model,train,test
0,MyLinearRegression_(sgd),1037.118346,1230.703180
1,MyLinearRegression_(gd),1469.141442,1479.425791
2,MyLinearRegression_(analytical),1035.351576,1226.832124
3,LinearRegression_(sklearn),1035.351576,1226.832124


In [14]:
RMSE

,model,train,test
0,MyLinearRegression_(sgd),1037.118346,1230.703180
1,MyLinearRegression_(gd),1469.141442,1479.425791
2,MyLinearRegression_(analytical),1035.351576,1226.832124
3,LinearRegression_(sklearn),1035.351576,1226.832124


In [15]:
MAE

,model,train,test
0,MyLinearRegression_(sgd),720.296321,725.320546
1,MyLinearRegression_(gd),1032.417158,1031.182190
2,MyLinearRegression_(analytical),711.791166,717.053857
3,LinearRegression_(sklearn),711.791166,717.053857


In [16]:
R2

,model,train,test
0,MyLinearRegression_(sgd),0.578599,0.401163
1,MyLinearRegression_(gd),0.154398,0.134657
2,MyLinearRegression_(analytical),0.580034,0.404924
3,LinearRegression_(sklearn),0.580034,0.404924


## 5. Regularized models implementation — Ridge, Lasso, ElasticNet

In [17]:
class MyRidge():
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.w = None

    def add_bias(self, X):
        return np.c_[np.ones(X.shape[0]), X]

    def fit(self, X, y):
        X = self.add_bias(X)
        n_features = X.shape[1]
        I = np.eye(n_features)
        self.w = np.linalg.inv(X.T @ X + self.alpha * I) @ X.T @ y

        return self

    def predict(self, X):
        X = self.add_bias(X)
        return X @ self.w
    
# для Lasso нельзя просто получить выражение, поэтому:
# 1) сначала фиксируем все веса, кроме одного
# 2) оптимизируем один w_j
# 3) повторяем

class MyLasso():
    def __init__(self, alpha=1.0, n_iter=100):
        self.alpha = alpha
        self.n_iter = n_iter
        self.w = None

    def add_bias(self, X):
        return np.c_[np.ones(X.shape[0]), X]

    def soft_threshold(self, z, lam):
        if z > lam:
            return z - lam
        elif z < -lam:
            return z + lam
        else:
            return 0

    def fit(self, X, y):
        X = self.add_bias(X)
        n_samples, n_features = X.shape

        self.w = np.zeros(n_features)

        for _ in range(self.n_iter):
            for j in range(n_features):

                residual = y - X @ self.w + self.w[j] * X[:, j]

                rho = np.dot(X[:, j], residual)

                if j == 0:
                    self.w[j] = rho / np.sum(X[:, j] ** 2)
                else:
                    self.w[j] = self.soft_threshold(
                        rho / np.sum(X[:, j] ** 2),
                        self.alpha
                    )

        return self

    def predict(self, X):
        X = self.add_bias(X)
        return X @ self.w
    

class MyElasticNet():
    def __init__(self, alpha=1.0, l1_ratio=0.5, n_iter=100):
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        self.n_iter = n_iter
        self.w = None

    def add_bias(self, X):
        return np.c_[np.ones(X.shape[0]), X]

    def soft_threshold(self, z, lam):
        if z > lam:
            return z - lam
        elif z < -lam:
            return z + lam
        else:
            return 0

    def fit(self, X, y):
        X = self.add_bias(X)
        n_samples, n_features = X.shape

        self.w = np.zeros(n_features)

        for _ in range(self.n_iter):
            for j in range(n_features):

                residual = y - X @ self.w + self.w[j] * X[:, j]

                rho = np.dot(X[:, j], residual)

                if j == 0:
                    self.w[j] = rho / np.sum(X[:, j] ** 2)
                else:
                    z = np.sum(X[:, j] ** 2)

                    self.w[j] = self.soft_threshold(
                        rho / z,
                        self.alpha * self.l1_ratio
                    ) / (1 + self.alpha * (1 - self.l1_ratio))

        return self

    def predict(self, X):
        X = self.add_bias(X)
        return X @ self.w

In [18]:
regularized_models = [
    ("MyRidge", MyRidge()),
    ("MyLasso", MyLasso()),
    ("MyElasticNet", MyElasticNet()),
    ("Ridge", Ridge()),
    ("Lasso", Lasso()),
    ("ElasticNet", ElasticNet()),
]

for model_name, model in regularized_models:
    fit_evaluate_add(results, model_name, model, X_train, y_train, X_test, y_test)

MAE, RMSE, R2 = sync_metric_tables(results)
RMSE

,model,train,test
0,MyLinearRegression_(sgd),1037.118346,1230.703180
1,MyLinearRegression_(gd),1469.141442,1479.425791
2,MyLinearRegression_(analytical),1035.351576,1226.832124
3,LinearRegression_(sklearn),1035.351576,1226.832124
4,MyRidge,1035.351580,1226.805454
5,MyLasso,1035.377238,1225.648179
6,MyElasticNet,1244.242661,1248.479217
7,Ridge,1035.351581,1226.789958
8,Lasso,1035.548472,1226.690472
9,ElasticNet,1189.928503,1204.449389


In [19]:
RMSE

,model,train,test
0,MyLinearRegression_(sgd),1037.118346,1230.703180
1,MyLinearRegression_(gd),1469.141442,1479.425791
2,MyLinearRegression_(analytical),1035.351576,1226.832124
3,LinearRegression_(sklearn),1035.351576,1226.832124
4,MyRidge,1035.351580,1226.805454
5,MyLasso,1035.377238,1225.648179
6,MyElasticNet,1244.242661,1248.479217
7,Ridge,1035.351581,1226.789958
8,Lasso,1035.548472,1226.690472
9,ElasticNet,1189.928503,1204.449389


In [20]:
MAE

,model,train,test
0,MyLinearRegression_(sgd),720.296321,725.320546
1,MyLinearRegression_(gd),1032.417158,1031.182190
2,MyLinearRegression_(analytical),711.791166,717.053857
3,LinearRegression_(sklearn),711.791166,717.053857
4,MyRidge,711.786292,717.048623
5,MyLasso,711.709218,716.951084
6,MyElasticNet,856.792782,858.523842
7,Ridge,711.787958,717.049975
8,Lasso,711.397503,716.636779
9,ElasticNet,807.091330,809.866492


In [21]:
R2

,model,train,test
0,MyLinearRegression_(sgd),0.578599,0.401163
1,MyLinearRegression_(gd),0.154398,0.134657
2,MyLinearRegression_(analytical),0.580034,0.404924
3,LinearRegression_(sklearn),0.580034,0.404924
4,MyRidge,0.580034,0.404950
5,MyLasso,0.580013,0.406072
6,MyElasticNet,0.393475,0.383739
7,Ridge,0.580034,0.404965
8,Lasso,0.579874,0.405062
9,ElasticNet,0.445272,0.426440


## 6. Feature normalization

Нормализация - необходимый этап подготовки данных для алгоритмов, чувствительных к масштабу признаков. К примеру, нам нужно обучить модель на данных, в которых в качестве признаков содержатся возраст человека (от 18 до 80) и уровень дохода (от 20000 до 200000). Без нормализации доход будет доминировать просто из-за своего масштаба, даже если он менее значим для задачи.

Но в то же время нормализация может ухудшить качество модели. Особенно в таких случаях, когда среди данных есть выбросы. К примеру, MinMaxScaler сжимает данные вокруг 0, и один выброс с очень высоким значением (например, зарплата в 10 миллионов рублей) может «сжать» остальные данные в узкий диапазон, уменьшив влияние других признаков. Также нормализация данных может ухудшить качество моделей для выявления аномалий (Isolation Forest).

Также есть некая середина, когда нормализация почти не влияет на качество обучения. Это часто прослеживается при обучении моделей DecisionTree и RandomForest.

### MinMaxScaler
- приводит каждый признак к диапазону от 0 до 1
$$
x_{scaled} = \frac{x - x_{min}}{x_{max} - x_{min}}
$$

### StandardScaler
- приводит данные к нормальному распределению с центром в нуле и стандартным отклонением, равным 1
$$
x_{scaled} = \frac{x - \mu}{\sigma}
$$

In [22]:
class MyMinMaxScaler():
    def __init__(self):
        self.min = None
        self.max = None

    def fit(self, X):
        X = np.asarray(X)
        self.min = X.min(axis=0)
        self.max = X.max(axis=0)
        return self
    
    def transform(self, X):
        X = np.asarray(X)
        data_range = self.max - self.min
        data_range[data_range == 0] = 1
        X_scaled = (X - self.min) / data_range
        return X_scaled
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)
    
class MyStandardScaler():
    def __init__(self):
        self.mean = None
        self.std = None

    def fit(self, X):
        X = np.asarray(X)
        self.mean = X.mean(axis=0)
        self.std = X.std(axis=0)
        return self
    
    def transform(self, X):
        X = np.asarray(X)
        X_scaled = (X - self.mean) / self.std
        return X_scaled
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)

In [23]:
scaler_compare = pd.DataFrame(columns=["scaler_name", "train_shape", "test_shape"])
scaled_features = {}
scalers = [MyMinMaxScaler(), MinMaxScaler(), MyStandardScaler(), StandardScaler()]

for scaler in scalers:
    X_train_scaled, X_test_scaled = scale_selected_columns(
        X_train,
        X_test,
        scaler,
        ["bathrooms", "bedrooms"],
    )
    scaler_name = scaler.__class__.__name__
    scaled_features[scaler_name] = (X_train_scaled, X_test_scaled)
    scaler_compare.loc[len(scaler_compare)] = [
        scaler_name,
        X_train_scaled.shape,
        X_test_scaled.shape,
    ]

scaler_compare

,scaler_name,train_shape,test_shape
0,MyMinMaxScaler,"(48379, 22)","(73255, 22)"
1,MinMaxScaler,"(48379, 22)","(73255, 22)"
2,MyStandardScaler,"(48379, 22)","(73255, 22)"
3,StandardScaler,"(48379, 22)","(73255, 22)"


## 7. Fit custom and sklearn models with normalized data

In [24]:
base_custom_models = [
    "MyLinearRegression_(analytical)",
    "MyRidge",
    "MyLasso",
    "MyElasticNet",
]
result_tables = select_metric_rows(results, base_custom_models)

normalized_models = [
    ("Linreg", MyLinearRegression(method="analytical", lr=0.001, n_iter=100)),
    ("Ridge", MyRidge()),
    ("Lasso", MyLasso()),
    ("ElasticNet", MyElasticNet()),
]

for scaler in [MyMinMaxScaler(), MyStandardScaler()]:
    X_train_scaled, X_test_scaled = scale_selected_columns(
        X_train,
        X_test,
        scaler,
        ["bathrooms", "bedrooms"],
    )
    scaler_name = scaler.__class__.__name__

    for model_label, model in normalized_models:
        fit_evaluate_add(
            result_tables,
            f"{model_label} {scaler_name}",
            model,
            X_train_scaled,
            y_train,
            X_test_scaled,
            y_test,
        )

result_MAE, result_RMSE, result_R2 = sync_metric_tables(result_tables)

In [25]:
result_RMSE.head(20)

,model,train,test
0,MyLinearRegression_(analytical),1035.351576,1226.832124
1,MyRidge,1035.351580,1226.805454
2,MyLasso,1035.377238,1225.648179
3,MyElasticNet,1244.242661,1248.479217
4,Linreg MyMinMaxScaler,1035.351576,1226.832124
5,Ridge MyMinMaxScaler,1035.383988,1222.000169
6,Lasso MyMinMaxScaler,1035.369684,1226.972287
7,ElasticNet MyMinMaxScaler,1243.913973,1248.190165
8,Linreg MyStandardScaler,1035.351576,1226.832124
9,Ridge MyStandardScaler,1035.351588,1226.821963


In [26]:
result_MAE.head(20)

,model,train,test
0,MyLinearRegression_(analytical),711.791166,717.053857
1,MyRidge,711.786292,717.048623
2,MyLasso,711.709218,716.951084
3,MyElasticNet,856.792782,858.523842
4,Linreg MyMinMaxScaler,711.791166,717.053857
5,Ridge MyMinMaxScaler,711.866773,717.054562
6,Lasso MyMinMaxScaler,711.685589,716.956928
7,ElasticNet MyMinMaxScaler,856.525318,858.260677
8,Linreg MyStandardScaler,711.791166,717.053857
9,Ridge MyStandardScaler,711.779711,717.042319


In [27]:
result_R2.head(20)

,model,train,test
0,MyLinearRegression_(analytical),0.580034,0.404924
1,MyRidge,0.580034,0.404950
2,MyLasso,0.580013,0.406072
3,MyElasticNet,0.393475,0.383739
4,Linreg MyMinMaxScaler,0.580034,0.404924
5,Ridge MyMinMaxScaler,0.580008,0.409603
6,Lasso MyMinMaxScaler,0.580019,0.404788
7,ElasticNet MyMinMaxScaler,0.393795,0.384025
8,Linreg MyStandardScaler,0.580034,0.404924
9,Ridge MyStandardScaler,0.580034,0.404934


## 8. Overfit models

- добавляем полиномиальные признаки

In [28]:
X_train_poly, X_test_poly, polynomial_features = make_polynomial_features(
    X_train,
    X_test,
    ["bathrooms", "bedrooms"],
    degree=10,
)

- нормализуем признаки

In [29]:
X_train_scaled, X_test_scaled, binary_cols, num_cols = scale_non_binary_features(
    X_train_poly,
    X_test_poly,
    StandardScaler(),
)

- обучение моделей

In [30]:
polynomial_models = [
    ("Linreg Polynomial", MyLinearRegression(method="analytical", lr=0.001, n_iter=100)),
    ("Ridge Polynomial", MyRidge(alpha=1000)),
    ("Lasso Polynomial", MyLasso(alpha=1000)),
    ("ElasticNet Polynomial", MyElasticNet(alpha=1000)),
]

for model_name, model in polynomial_models:
    fit_evaluate_add(
        result_tables,
        model_name,
        model,
        X_train_scaled,
        y_train,
        X_test_scaled,
        y_test,
    )

result_MAE, result_RMSE, result_R2 = sync_metric_tables(result_tables)

In [31]:
result_MAE

,model,train,test
0,MyLinearRegression_(analytical),711.791166,7.170539e+02
1,MyRidge,711.786292,7.170486e+02
2,MyLasso,711.709218,7.169511e+02
3,MyElasticNet,856.792782,8.585238e+02
4,Linreg MyMinMaxScaler,711.791166,7.170539e+02
5,Ridge MyMinMaxScaler,711.866773,7.170546e+02
6,Lasso MyMinMaxScaler,711.685589,7.169569e+02
7,ElasticNet MyMinMaxScaler,856.525318,8.582607e+02
8,Linreg MyStandardScaler,711.791166,7.170539e+02
9,Ridge MyStandardScaler,711.779711,7.170423e+02


In [32]:
result_R2

,model,train,test
0,MyLinearRegression_(analytical),0.580034,4.049244e-01
1,MyRidge,0.580034,4.049503e-01
2,MyLasso,0.580013,4.060724e-01
3,MyElasticNet,0.393475,3.837393e-01
4,Linreg MyMinMaxScaler,0.580034,4.049244e-01
5,Ridge MyMinMaxScaler,0.580008,4.096027e-01
6,Lasso MyMinMaxScaler,0.580019,4.047885e-01
7,ElasticNet MyMinMaxScaler,0.393795,3.840246e-01
8,Linreg MyStandardScaler,0.580034,4.049244e-01
9,Ridge MyStandardScaler,0.580034,4.049343e-01


In [33]:
result_RMSE

,model,train,test
0,MyLinearRegression_(analytical),1035.351576,1.226832e+03
1,MyRidge,1035.351580,1.226805e+03
2,MyLasso,1035.377238,1.225648e+03
3,MyElasticNet,1244.242661,1.248479e+03
4,Linreg MyMinMaxScaler,1035.351576,1.226832e+03
5,Ridge MyMinMaxScaler,1035.383988,1.222000e+03
6,Lasso MyMinMaxScaler,1035.369684,1.226972e+03
7,ElasticNet MyMinMaxScaler,1243.913973,1.248190e+03
8,Linreg MyStandardScaler,1035.351576,1.226832e+03
9,Ridge MyStandardScaler,1035.351588,1.226822e+03


## 9. Native models

In [34]:
mean = y_train.mean()
add_baseline_result(result_tables, "Naive mean", mean, mean, y_train, y_test)

median = y_train.median()
add_baseline_result(result_tables, "Naive median", median, median, y_train, y_test)

result_MAE, result_RMSE, result_R2 = sync_metric_tables(result_tables)

## 10. Compare results

In [35]:
result_MAE

,model,train,test
0,MyLinearRegression_(analytical),711.791166,7.170539e+02
1,MyRidge,711.786292,7.170486e+02
2,MyLasso,711.709218,7.169511e+02
3,MyElasticNet,856.792782,8.585238e+02
4,Linreg MyMinMaxScaler,711.791166,7.170539e+02
5,Ridge MyMinMaxScaler,711.866773,7.170546e+02
6,Lasso MyMinMaxScaler,711.685589,7.169569e+02
7,ElasticNet MyMinMaxScaler,856.525318,8.582607e+02
8,Linreg MyStandardScaler,711.791166,7.170539e+02
9,Ridge MyStandardScaler,711.779711,7.170423e+02


In [36]:
result_RMSE

,model,train,test
0,MyLinearRegression_(analytical),1035.351576,1.226832e+03
1,MyRidge,1035.351580,1.226805e+03
2,MyLasso,1035.377238,1.225648e+03
3,MyElasticNet,1244.242661,1.248479e+03
4,Linreg MyMinMaxScaler,1035.351576,1.226832e+03
5,Ridge MyMinMaxScaler,1035.383988,1.222000e+03
6,Lasso MyMinMaxScaler,1035.369684,1.226972e+03
7,ElasticNet MyMinMaxScaler,1243.913973,1.248190e+03
8,Linreg MyStandardScaler,1035.351576,1.226832e+03
9,Ridge MyStandardScaler,1035.351588,1.226822e+03


In [37]:
result_R2

,model,train,test
0,MyLinearRegression_(analytical),0.580034,4.049244e-01
1,MyRidge,0.580034,4.049503e-01
2,MyLasso,0.580013,4.060724e-01
3,MyElasticNet,0.393475,3.837393e-01
4,Linreg MyMinMaxScaler,0.580034,4.049244e-01
5,Ridge MyMinMaxScaler,0.580008,4.096027e-01
6,Lasso MyMinMaxScaler,0.580019,4.047885e-01
7,ElasticNet MyMinMaxScaler,0.393795,3.840246e-01
8,Linreg MyStandardScaler,0.580034,4.049244e-01
9,Ridge MyStandardScaler,0.580034,4.049343e-01


Вывод: лучшая модель - Lasso Regression (Standard Scaler), но тем не менее Lasso Polynomial (Standard Scaler) является более стабильной. Метрика R2 у данной модели на train и test данных отличается на 0,002375.